Codigo del tfg de David García de Toledo

Primero vamos a obtener todos los datos historicos mediante y y finance

In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
 
 
# ============================================================
# CONFIGURACIÓN
# ============================================================
 
# Diccionario {nombre_activo: ticker_yfinance}
ACTIVOS = {
    "sp500": "^GSPC",   # Índice S&P 500
    "aapl":  "AAPL",    # Apple Inc.
    "msft":  "MSFT",    # Microsoft Corp.
    "gold":  "GC=F",    # Futuro del oro (COMEX)
}
 
PERIODO = "10y"             # Periodo de histórico
VENTANA_VAR = 21            # Ventana móvil para la varianza (días)
DIAS_ANIO = 252             # Días de trading por año (anualización)
CARPETA_SALIDA = "datos"    # Carpeta donde se guardan los CSV
 
 
# ============================================================
# FUNCIONES
# ============================================================
 
def descargar_activo(nombre: str, ticker: str, periodo: str) -> pd.DataFrame:
    """
    Descarga los datos históricos de un activo desde Yahoo Finance
    y añade columnas con retornos logarítmicos y varianza histórica
    anualizada.
    """
    print(f"  - Descargando {nombre.upper()} ({ticker})...")
 
    df = yf.download(
        ticker,
        period=periodo,
        auto_adjust=False,
        progress=False,
    )
 
    # En versiones recientes de yfinance las columnas vienen como MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
 
    df = df.dropna()
 
    # Columna de precio que se usará para retornos:
    # se prefiere "Adj Close" si está disponible; si no, "Close"
    col_precio = "Adj Close" if "Adj Close" in df.columns else "Close"
 
    # Retorno logarítmico diario
    df["Log_Return"] = np.log(df[col_precio] / df[col_precio].shift(1))
 
    # Varianza histórica anualizada (ventana móvil de 21 días)
    df["Var_Hist"] = df["Log_Return"].rolling(VENTANA_VAR).var() * DIAS_ANIO
 
    df = df.dropna()
    return df
 
 
def guardar_csv(df: pd.DataFrame, nombre: str, carpeta: str) -> str:
    """
    Guarda el DataFrame en un CSV dentro de la carpeta indicada
    y devuelve la ruta completa al archivo.
    """
    os.makedirs(carpeta, exist_ok=True)
    ruta = os.path.join(carpeta, f"{nombre}.csv")
    df.to_csv(ruta, index=True)
    return ruta
 
 
# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================
 
def main():
    print("=" * 60)
    print("Descarga de datos históricos - Modelo de Heston")
    print("=" * 60)
    print(f"Periodo:        {PERIODO}")
    print(f"Activos:        {list(ACTIVOS.keys())}")
    print(f"Carpeta salida: {CARPETA_SALIDA}/")
    print("-" * 60)
 
    resumen = []
 
    for nombre, ticker in ACTIVOS.items():
        df = descargar_activo(nombre, ticker, PERIODO)
        ruta = guardar_csv(df, nombre, CARPETA_SALIDA)
 
        resumen.append({
            "activo":      nombre.upper(),
            "ticker":      ticker,
            "filas":       len(df),
            "desde":       df.index[0].strftime("%Y-%m-%d"),
            "hasta":       df.index[-1].strftime("%Y-%m-%d"),
            "var_media":   df["Var_Hist"].mean(),
            "ruta":        ruta,
        })
 
    print("-" * 60)
    print("Resumen:")
    df_resumen = pd.DataFrame(resumen)
    print(df_resumen.to_string(index=False))
    print("=" * 60)
    print("Descarga completada correctamente.")
 
 
if __name__ == "__main__":
    main()

### Regresión lineal

## Ya tenemos los datos descargados de Yfinance, ahora toca, regresion lineal, ==============================================================
 CALIBRACIÓN DEL MODELO DE HESTON POR REGRESIÓN LINEAL
==============================================================

Este script estima los parámetros del modelo de Heston

    dS_t   = mu * S_t * dt + sqrt(v_t) * S_t * dW^S_t
    dv_t   = kappa * (theta - v_t) * dt + xi * sqrt(v_t) * dW^v_t

para los cuatro activos del trabajo (S&P 500, Apple, Microsoft
y oro) a partir de los CSVs generados por 'descargar_datos.py'.

La estimación se basa en la discretización del proceso CIR:

    Delta v_t  =  kappa * (theta - v_t) * dt  +  ruido

que reescrita como

    Delta v_t  =  (kappa * theta * dt)  -  (kappa * dt) * v_t  +  ruido

es una regresión lineal de Delta v_t sobre v_t. De los
coeficientes se recuperan kappa y theta; xi se obtiene de los
residuos y rho de la correlación entre retornos y residuos.

Autor:    David García de Toledo
TFG:      Calibración del modelo de Heston
==============================================================
"""

In [ ]:


import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression


# ============================================================
# CONFIGURACIÓN
# ============================================================

ACTIVOS = ["sp500", "aapl", "msft", "gold"]

CARPETA_DATOS     = "datos"          # Donde están los CSVs de entrada
CARPETA_RESULTADOS = "resultados"    # Donde se guardan los parámetros

DT        = 1 / 252      # Paso temporal (1 día de trading)
DIAS_ANIO = 252          # Días de trading por año


# ============================================================
# FUNCIONES
# ============================================================

def cargar_datos(nombre: str, carpeta: str) -> pd.DataFrame:
    """
    Carga el CSV de un activo y devuelve un DataFrame con las
    columnas necesarias: Log_Return y Var_Hist.
    """
    ruta = os.path.join(carpeta, f"{nombre}.csv")
    df = pd.read_csv(ruta, index_col=0, parse_dates=True)
    df = df.dropna(subset=["Log_Return", "Var_Hist"])
    return df


def calibrar_heston(df: pd.DataFrame, dt: float = DT) -> dict:
    """
    Estima los cinco parámetros del modelo de Heston a partir
    de los retornos logarítmicos y la varianza histórica
    anualizada del activo.

    Devuelve un diccionario con:
        kappa : velocidad de reversión
        theta : nivel de largo plazo de la varianza
        xi    : volatilidad de la volatilidad
        rho   : correlación precio-volatilidad
        v0    : varianza inicial (la más reciente)
        R2    : coeficiente de determinación de la regresión
    """
    var_hist   = df["Var_Hist"].values
    log_return = df["Log_Return"].values

    # Incrementos de la varianza
    v_t        = var_hist[:-1]
    delta_v    = var_hist[1:] - v_t

    # --- Regresión lineal: Delta v_t = a + b * v_t ---
    X = v_t.reshape(-1, 1)
    Y = delta_v
    modelo = LinearRegression().fit(X, Y)

    intercept = modelo.intercept_
    pendiente = modelo.coef_[0]

    # --- Recuperación de kappa y theta ---
    #   pendiente = -kappa * dt   =>  kappa = -pendiente / dt
    #   intercept =  kappa * theta * dt  =>  theta = intercept / (kappa * dt)
    kappa = -pendiente / dt
    kappa = max(kappa, 1e-4)           # Aseguramos positividad
    theta = intercept / (kappa * dt)
    theta = max(theta, 1e-4)           # Aseguramos positividad

    # --- xi: volatilidad de la volatilidad ---
    residuos = delta_v - modelo.predict(X)
    xi = np.std(residuos) / np.sqrt(np.mean(v_t) * dt)

    # --- rho: correlación entre retornos y residuos de la varianza ---
    log_ret_alineado = log_return[-len(residuos):]
    rho = np.corrcoef(log_ret_alineado, residuos)[0, 1]

    # --- v0: varianza inicial (último valor observado) ---
    v0 = var_hist[-1]

    # --- R² de la regresión ---
    r2 = modelo.score(X, Y)

    return {
        "kappa": kappa,
        "theta": theta,
        "xi":    xi,
        "rho":   rho,
        "v0":    v0,
        "R2":    r2,
    }


def imprimir_parametros(nombre: str, params: dict) -> None:
    """Imprime los parámetros calibrados de un activo."""
    print(f"\n--- Parámetros de Heston para {nombre.upper()} ---")
    print(f"  κ  (Velocidad de reversión)       : {params['kappa']:.4f}")
    print(f"  θ  (Varianza a largo plazo)       : {params['theta']:.4f}")
    print(f"  ξ  (Volatilidad de la volatilidad): {params['xi']:.4f}")
    print(f"  ρ  (Correlación precio-vol)       : {params['rho']:.4f}")
    print(f"  v₀ (Varianza inicial)             : {params['v0']:.4f}")
    print(f"  R² de la regresión                : {params['R2']:.4f}")


def guardar_resultados(resultados: list, carpeta: str) -> str:
    """
    Guarda los parámetros calibrados de todos los activos en un
    único CSV y devuelve la ruta del archivo.
    """
    os.makedirs(carpeta, exist_ok=True)
    ruta = os.path.join(carpeta, "parametros_regresion_lineal.csv")
    df = pd.DataFrame(resultados)
    df.to_csv(ruta, index=False)
    return ruta


# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    print("=" * 64)
    print("Calibración del modelo de Heston por regresión lineal")
    print("=" * 64)
    print(f"Activos:           {ACTIVOS}")
    print(f"Paso temporal dt:  1/{DIAS_ANIO}")
    print("-" * 64)

    resultados = []

    for nombre in ACTIVOS:
        df = cargar_datos(nombre, CARPETA_DATOS)
        params = calibrar_heston(df)
        imprimir_parametros(nombre, params)

        resultados.append({
            "activo": nombre.upper(),
            **params,
        })

    print("-" * 64)
    ruta = guardar_resultados(resultados, CARPETA_RESULTADOS)
    print(f"Resultados guardados en: {ruta}")
    print("=" * 64)


if __name__ == "__main__":
    main()

## Conclusión de los resultados de parametros.



R2 de la regresión: NO está en la ecuación de Heston.
El R2R^2
R2 no es un parámetro del modelo. Es una métrica de calidad de la estimación que indica cómo de bien la regresión lineal está capturando la relación entre los incrementos de la varianza Δνt\Delta\nu_t
Δνt​ y la varianza actual νt\nu_t
νt​.

R2≈1R^2 \approx 1
R2≈1: la relación es bien lineal, la regresión funciona perfectamente
R2≈0R^2 \approx 0
R2≈0: la regresión apenas explica nada de los datos, la relación no es lineal o hay mucho ruido

Tus resultados dan R2R^2
R2 entre 0.003 y 0.009, es decir, menos del 1%. Esto significa que la regresión lineal capta muy poca señal de los datos. Y esto es precisamente el argumento del TFG: como la relación no es bien lineal, una red neuronal con capacidad no lineal debería ser capaz de captar lo que la regresión pierde.
Por tanto el R2R^2
R2 no entra en la simulación ni se usa para predecir. Es solo una métrica diagnóstica que va en el TFG para justificar la motivación del trabajo. Si quieres lo quito del CSV final para no liarte y lo dejo solo como impresión por pantalla.

### Red Neuronal

Calibración del modelo de Heston mediante una **red neuronal que aprende el mapa inverso** del proceso CIR.

A diferencia de la regresión lineal (solución cerrada), aquí se entrena un perceptrón multicapa (MLP) para que, dada la trayectoria de varianza de un activo, devuelva los parámetros que con mayor verosimilitud la generaron:

1. Se simulan miles de trayectorias CIR con parámetros (kappa, theta, xi) **conocidos** (datos sintéticos = solo material de entrenamiento).
2. Cada trayectoria se resume en **9 características** que separan el *nivel* (-> theta) de la *forma* (-> kappa, xi).
3. La red aprende a recuperar (corrección de theta, log kappa, log xi) a partir de esas características.
4. La red ya entrenada se aplica a las **series reales** de los 4 activos (los resultados NO salen de datos inventados).

**Detalles de diseño:**
- **theta anclado a la media:** la red predice una corrección logarítmica acotada con tanh (margen +-0.25), de modo que `theta = media * exp(correccion)`. Así theta no puede alejarse del nivel observado, que es lo que theta ES por definición.
- **kappa y xi libres**, pero **kappa se recorta al rango de entrenamiento [0.5, 8]** tras la inferencia: fuera de ese rango la red extrapola sin soporte y produce valores sin sentido (kappa desorbitado, R2 del drift negativo). La velocidad de reversión es notoriamente difícil de identificar con una sola trayectoria, así que se restringe al dominio en el que la red fue entrenada.
- **rho, v0 y mu** se calculan directamente sobre los datos, igual que en la regresión.

**Arquitectura:** 9 -> 128 -> 128 -> 128 -> 3, ReLU, Adam (lr=1e-3), MSELoss, 6000 muestras sintéticas, 3000 épocas, semilla 42.

Autor: David García de Toledo &mdash; TFG: Calibración del modelo de Heston

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# CONFIGURACIÓN
# ============================================================

ACTIVOS = ["sp500", "aapl", "msft", "gold"]

CARPETA_DATOS      = "datos"
CARPETA_RESULTADOS = "resultados"

DT        = 1 / 252
DIAS_ANIO = 252

# Hiperparámetros de la red y del entrenamiento
N_MUESTRAS_ENTRENO = 6000     # trayectorias sintéticas de entrenamiento
LONGITUD_SERIE     = 2500     # longitud de cada trayectoria (~10 años)
N_EPOCAS           = 3000
LEARNING_RATE      = 1e-3
N_OCULTAS          = 128      # neuronas por capa oculta
MARGEN_THETA       = 0.25     # margen del nivel de largo plazo en torno a la media
SEMILLA            = 42

torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)


# ============================================================
# SIMULADOR DEL PROCESO CIR (genera los datos de entrenamiento)
# ============================================================

def simular_cir_batch(kappa, theta, xi, v0, n, dt=DT, rng=None):
    """Simula len(kappa) trayectorias CIR en paralelo. Devuelve (N, n)."""
    if rng is None:
        rng = np.random.default_rng(0)
    N = len(kappa)
    v = np.empty((N, n))
    v[:, 0] = v0
    sq_dt = np.sqrt(dt)
    for t in range(1, n):
        prev = v[:, t - 1]
        z = rng.standard_normal(N)
        dv = kappa * (theta - prev) * dt + xi * np.sqrt(np.maximum(prev, 0)) * sq_dt * z
        v[:, t] = np.maximum(prev + dv, 1e-8)
    return v


# ============================================================
# EXTRACCIÓN DE CARACTERÍSTICAS DE UNA TRAYECTORIA DE VARIANZA
# ------------------------------------------------------------
# El nivel (logaritmo de la media) se separa de la forma: las demás
# características son invariantes de escala, de modo que los picos de
# la serie afectan a la forma (kappa, xi) pero no al nivel (theta).
# ============================================================

def autocorr_batch(V, lag):
    a, b = V[:, :-lag], V[:, lag:]
    am, bm = a.mean(1, keepdims=True), b.mean(1, keepdims=True)
    num = ((a - am) * (b - bm)).sum(1)
    den = np.sqrt(((a - am) ** 2).sum(1) * ((b - bm) ** 2).sum(1)) + 1e-12
    return num / den


def features_batch(V):
    """Devuelve (caracteristicas, log_media) para cada trayectoria (por filas)."""
    V = V.astype(np.float64)
    dv = np.diff(V, axis=1)
    v_t = V[:, :-1]
    n = v_t.shape[1]
    m = V.mean(1)
    sx, sy = v_t.sum(1), dv.sum(1)
    sxx, sxy = (v_t ** 2).sum(1), (v_t * dv).sum(1)
    pend = (n * sxy - sx * sy) / (n * sxx - sx ** 2 + 1e-18)
    corte = (sy - pend * sx) / n
    resid = dv - (pend[:, None] * v_t + corte[:, None])
    log_m = np.log(m)
    X = np.column_stack([
        log_m,                        # nivel  -> theta
        pend,                         # pendiente (tasa) -> kappa
        resid.std(1) / np.sqrt(m),    # ~ xi*sqrt(dt)    -> xi
        V.std(1) / m,                 # coeficiente de variación (forma)
        dv.std(1) / m,                # dispersión de incrementos (forma)
        autocorr_batch(V, 1), autocorr_batch(V, 5),
        autocorr_batch(V, 10), autocorr_batch(V, 20),
    ])
    return X, log_m


def features_serie(v):
    """Características y log-media de una única serie."""
    X, log_m = features_batch(v.reshape(1, -1))
    return X[0], log_m[0]


# ============================================================
# GENERACIÓN DEL CONJUNTO DE ENTRENAMIENTO SINTÉTICO
# ============================================================

def generar_dataset(n_muestras, n=LONGITUD_SERIE, rng=None):
    """
    Muestrea parámetros (kappa, theta, xi) en rangos amplios, simula una
    trayectoria CIR con cada conjunto (arrancando en el nivel de largo plazo)
    y extrae sus características. El objetivo del nivel se expresa como la
    corrección logarítmica respecto de la media observada, acotada al margen.
    """
    if rng is None:
        rng = np.random.default_rng(0)
    kappa = np.exp(rng.uniform(np.log(0.5),   np.log(8.0),  n_muestras))
    theta = np.exp(rng.uniform(np.log(0.005), np.log(0.20), n_muestras))
    xi    = np.exp(rng.uniform(np.log(0.2),   np.log(1.5),  n_muestras))
    V = simular_cir_batch(kappa, theta, xi, theta, n, rng=rng)
    X, log_m = features_batch(V)
    correccion = np.clip(np.log(theta) - log_m, -MARGEN_THETA, MARGEN_THETA)
    Y = np.column_stack([correccion, np.log(kappa), np.log(xi)])
    return X, Y


# ============================================================
# RED NEURONAL CALIBRADORA
# ============================================================

class RedCalibradora(nn.Module):
    """
    Perceptrón multicapa (MLP) feed-forward.
    Arquitectura: 9 (entradas) -> 128 -> 128 -> 128 -> 3 (salidas), ReLU.
    Salida: (corrección de theta acotada, log kappa, log xi). El nivel de
    largo plazo se obtiene como theta = media * exp(corrección), de modo que
    no puede alejarse del nivel observado en la serie.
    """
    def __init__(self, n_in=9, h=N_OCULTAS, n_out=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, h), nn.ReLU(),
            nn.Linear(h, h),    nn.ReLU(),
            nn.Linear(h, h),    nn.ReLU(),
            nn.Linear(h, n_out),
        )

    def forward(self, x):
        o = self.net(x)
        # canal 0 (corrección de theta) acotado; canales 1, 2 (log kappa, log xi) libres
        return torch.cat([MARGEN_THETA * torch.tanh(o[:, :1]), o[:, 1:]], dim=1)


def entrenar_red():
    """Genera el dataset sintético y entrena la red. Devuelve red y normalización."""
    torch.manual_seed(SEMILLA)
    np.random.seed(SEMILLA)

    print("Generando conjunto de entrenamiento sintético...")
    X_train, Y_train = generar_dataset(N_MUESTRAS_ENTRENO,
                                        rng=np.random.default_rng(SEMILLA))
    X_mean, X_std = X_train.mean(0), X_train.std(0)

    Xn = np.clip((X_train - X_mean) / X_std, -5, 5)
    Xt = torch.tensor(Xn, dtype=torch.float64)
    Yt = torch.tensor(Y_train, dtype=torch.float64)

    red = RedCalibradora().double()
    opt = optim.Adam(red.parameters(), lr=LEARNING_RATE)
    lossf = nn.MSELoss()

    print(f"Entrenando la red ({N_EPOCAS} épocas)...")
    for epoch in range(N_EPOCAS):
        opt.zero_grad()
        loss = lossf(red(Xt), Yt)
        loss.backward()
        opt.step()
        if epoch % 1000 == 0:
            print(f"  época {epoch:5d}   pérdida {loss.item():.5f}")
    print(f"  pérdida final: {loss.item():.5f}")
    return red, X_mean, X_std


# ============================================================
# CALIBRACIÓN DE UN ACTIVO REAL CON LA RED YA ENTRENADA
# ============================================================

def cargar_datos(nombre, carpeta):
    ruta = os.path.join(carpeta, f"{nombre}.csv")
    df = pd.read_csv(ruta, index_col=0, parse_dates=True)
    return df.dropna(subset=["Log_Return", "Var_Hist"])


def calibrar_activo(df, red, X_mean, X_std, dt=DT):
    var_hist   = df["Var_Hist"].values.astype(np.float64)
    log_return = df["Log_Return"].values.astype(np.float64)

    # 1) La red estima theta (anclado a la media), kappa y xi
    fvec, log_m = features_serie(var_hist)
    f = np.clip((fvec - X_mean) / X_std, -5, 5)
    with torch.no_grad():
        o = red(torch.tensor(f, dtype=torch.float64).reshape(1, -1)).numpy()[0]
    theta = float(np.exp(o[0] + log_m))
    kappa = float(np.clip(np.exp(o[1]), 0.5, 8.0))  # se restringe al rango de entrenamiento de la red (extrapolar fuera no tiene soporte)
    xi    = float(np.exp(o[2]))

    # 2) rho: correlación entre retornos y residuos del drift CIR
    v_t = var_hist[:-1]
    delta_v = var_hist[1:] - v_t
    residuos = delta_v - kappa * (theta - v_t) * dt
    log_ret_alineado = log_return[-len(residuos):]
    rho = float(np.corrcoef(log_ret_alineado, residuos)[0, 1])

    # 3) v0: última varianza observada
    v0 = float(var_hist[-1])

    # 4) R² del ajuste del drift con los parámetros estimados
    ss_res = np.sum(residuos ** 2)
    ss_tot = np.sum((delta_v - delta_v.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot

    return {"kappa": kappa, "theta": theta, "xi": xi,
            "rho": rho, "v0": v0, "R2": float(r2)}


def imprimir_parametros(nombre, p):
    print(f"\n--- Parametros de Heston para {nombre.upper()} (red neuronal) ---")
    print(f"  kappa (Velocidad de reversion)       : {p['kappa']:.4f}")
    print(f"  theta (Varianza a largo plazo)       : {p['theta']:.4f}")
    print(f"  xi    (Volatilidad de la volatilidad): {p['xi']:.4f}")
    print(f"  rho   (Correlacion precio-vol)       : {p['rho']:.4f}")
    print(f"  v0    (Varianza inicial)             : {p['v0']:.4f}")
    print(f"  R2 del ajuste de la deriva           : {p['R2']:.4f}")


def guardar_resultados(resultados, carpeta):
    os.makedirs(carpeta, exist_ok=True)
    ruta = os.path.join(carpeta, "parametros_red_neuronal.csv")
    pd.DataFrame(resultados).to_csv(ruta, index=False)
    return ruta


# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    print("=" * 64)
    print("Calibracion del modelo de Heston por RED NEURONAL")
    print("=" * 64)
    print(f"Activos:               {ACTIVOS}")
    print(f"Muestras de entreno:   {N_MUESTRAS_ENTRENO}")
    print(f"Arquitectura:          9 -> {N_OCULTAS} -> {N_OCULTAS} -> {N_OCULTAS} -> 3 (ReLU)")
    print(f"Optimizador:           Adam (lr={LEARNING_RATE})")
    print("-" * 64)

    red, X_mean, X_std = entrenar_red()

    print("-" * 64)
    print("Aplicando la red a las series reales:")
    resultados = []
    for nombre in ACTIVOS:
        df = cargar_datos(nombre, CARPETA_DATOS)
        params = calibrar_activo(df, red, X_mean, X_std)
        imprimir_parametros(nombre, params)
        resultados.append({"activo": nombre.upper(), **params})

    print("-" * 64)
    ruta = guardar_resultados(resultados, CARPETA_RESULTADOS)
    print(f"Resultados guardados en: {ruta}")
    print("=" * 64)


if __name__ == "__main__":
    main()


**Sobre los parámetros obtenidos.** theta y xi salen parecidos a los de la regresión: theta por el anclaje a la media, y xi porque ambos métodos leen la misma dispersión de la varianza. La diferencia real está en **kappa**: red y regresión pueden discrepar porque la velocidad de reversión es difícil de identificar a partir de una única trayectoria. El recorte de kappa al rango de entrenamiento evita los valores extremos que aparecían al extrapolar (kappa de 30-47 y R2 negativos).

Como la trayectoria central del precio la gobierna **mu** (idéntico en ambos métodos) y apenas depende de kappa, las predicciones de ambos métodos resultan muy parecidas; las diferencias en kappa y xi afectan sobre todo a la **anchura de las bandas** de incertidumbre, no a la media. Eso permite responder a la pregunta del TFG ("¿cuál predice mejor?") de forma empírica.

## PRIMERO, SIMULAMOS CON REGRESIÓN LINEAL

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
 
 
# ============================================================
# CONFIGURACIÓN
# ============================================================
 
ACTIVOS = ["sp500", "aapl", "msft", "gold"]
 
CARPETA_DATOS      = "datos"
CARPETA_RESULTADOS = "resultados"
CARPETA_FIGURAS    = "figuras/regresion_lineal"
 
# Hiperparámetros de la simulación
DT          = 1 / 252
DIAS_ANIO   = 252
HORIZONTE   = 252           # 1 año de trading
N_SIMS      = 1000          # Número de trayectorias Monte Carlo
TEST_DAYS   = 252           # Días para comparar con datos reales (si existen)
SEMILLA     = 42
 
# Drift mu: se estima de la media de retornos históricos
# (lo definimos por activo dentro del bucle)
 
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})
 
 
# ============================================================
# FUNCIONES
# ============================================================
 
def cargar_datos(nombre: str, carpeta: str) -> pd.DataFrame:
    """Carga el CSV de un activo y devuelve un DataFrame limpio."""
    ruta = os.path.join(carpeta, f"{nombre}.csv")
    df = pd.read_csv(ruta, index_col=0, parse_dates=True)
    df = df.dropna(subset=["Log_Return", "Var_Hist"])
    return df
 
 
def cargar_parametros(carpeta: str, archivo: str) -> pd.DataFrame:
    """Carga los parámetros calibrados desde un CSV."""
    ruta = os.path.join(carpeta, archivo)
    return pd.read_csv(ruta)
 
 
def simular_heston(
    s0: float,
    v0: float,
    mu: float,
    kappa: float,
    theta: float,
    xi: float,
    rho: float,
    dt: float,
    horizonte: int,
    n_sims: int,
    semilla: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Simula n_sims trayectorias del modelo de Heston durante
    'horizonte' pasos temporales mediante el esquema de
    Euler-Maruyama con truncamiento para mantener la varianza
    no negativa.
 
    Devuelve:
        S : array (n_sims, horizonte+1) con las trayectorias del precio
        V : array (n_sims, horizonte+1) con las trayectorias de la varianza
    """
    rng = np.random.default_rng(semilla)
 
    S = np.zeros((n_sims, horizonte + 1))
    V = np.zeros((n_sims, horizonte + 1))
    S[:, 0] = s0
    V[:, 0] = v0
 
    # Brownianos correlacionados
    Z1 = rng.standard_normal((n_sims, horizonte))
    Z2_indep = rng.standard_normal((n_sims, horizonte))
    Z2 = rho * Z1 + np.sqrt(1 - rho**2) * Z2_indep
 
    sqrt_dt = np.sqrt(dt)
 
    for t in range(horizonte):
        v_prev = np.maximum(V[:, t], 0.0)
        sqrt_v = np.sqrt(v_prev)
 
        # Varianza (con truncamiento para evitar valores negativos)
        V[:, t+1] = np.maximum(
            v_prev + kappa * (theta - v_prev) * dt
            + xi * sqrt_v * sqrt_dt * Z1[:, t],
            0.0,
        )
 
        # Precio
        S[:, t+1] = S[:, t] * np.exp(
            (mu - 0.5 * v_prev) * dt + sqrt_v * sqrt_dt * Z2[:, t]
        )
 
    return S, V
 
 
def plot_trayectorias_varianza(
    V: np.ndarray, theta: float, nombre: str, carpeta: str
) -> str:
    """Gráfica de trayectorias de la varianza simuladas."""
    fig, ax = plt.subplots(figsize=(11, 5))
 
    # Algunas trayectorias individuales (alpha bajo)
    n_mostrar = min(100, V.shape[0])
    for i in range(n_mostrar):
        ax.plot(V[i], lw=0.5, alpha=0.15, color="steelblue")
 
    # Media y bandas de confianza
    media = V.mean(axis=0)
    p5    = np.percentile(V, 5, axis=0)
    p95   = np.percentile(V, 95, axis=0)
 
    ax.plot(media, color="navy", lw=2, label="Media simulada")
    ax.fill_between(range(len(media)), p5, p95, color="navy", alpha=0.15,
                    label="Intervalo de confianza 90%")
    ax.axhline(theta, color="red", linestyle="--", lw=1.5,
               label=fr"$\theta$ = {theta:.4f}")
 
    ax.set_title(f"Trayectorias simuladas de la varianza  v_t  -  {nombre.upper()}")
    ax.set_xlabel("Días de trading")
    ax.set_ylabel("Varianza anualizada")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
 
    ruta = os.path.join(carpeta, f"{nombre}_varianza.png")
    fig.savefig(ruta)
    plt.close(fig)
    return ruta
 
 
def plot_trayectorias_precio(
    S: np.ndarray,
    s0: float,
    nombre: str,
    carpeta: str,
    precio_real: np.ndarray | None = None,
) -> str:
    """Gráfica de trayectorias del precio simuladas."""
    fig, ax = plt.subplots(figsize=(11, 5))
 
    # Trayectorias individuales
    n_mostrar = min(100, S.shape[0])
    for i in range(n_mostrar):
        ax.plot(S[i], lw=0.5, alpha=0.15, color="forestgreen")
 
    # Media y bandas de confianza
    media = S.mean(axis=0)
    p5    = np.percentile(S, 5, axis=0)
    p95   = np.percentile(S, 95, axis=0)
 
    ax.plot(media, color="darkgreen", lw=2, label="Media simulada")
    ax.fill_between(range(len(media)), p5, p95, color="darkgreen", alpha=0.15,
                    label="Intervalo de confianza 90%")
 
    # Precio real si está disponible
    if precio_real is not None:
        ax.plot(precio_real, color="black", lw=2, label="Precio real observado")
 
    ax.axhline(s0, color="gray", linestyle=":", lw=1, alpha=0.6,
               label=f"Precio inicial S₀ = {s0:.2f}")
 
    ax.set_title(f"Trayectorias simuladas del precio  S_t  -  {nombre.upper()}")
    ax.set_xlabel("Días de trading")
    ax.set_ylabel("Precio")
    ax.legend(loc="best")
    ax.grid(alpha=0.3)
 
    ruta = os.path.join(carpeta, f"{nombre}_precio.png")
    fig.savefig(ruta)
    plt.close(fig)
    return ruta
 
 
def plot_distribucion_final(
    S: np.ndarray, s0: float, nombre: str, carpeta: str
) -> str:
    """Histograma de la distribución del precio al final del horizonte."""
    precio_final = S[:, -1]
 
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.hist(precio_final, bins=60, density=True,
            color="cornflowerblue", edgecolor="white", alpha=0.85)
 
    # Líneas de referencia
    media   = precio_final.mean()
    mediana = np.median(precio_final)
    p5      = np.percentile(precio_final, 5)
    p95     = np.percentile(precio_final, 95)
 
    ax.axvline(s0,      color="black", linestyle=":",  lw=1.5, label=f"S₀ = {s0:.2f}")
    ax.axvline(media,   color="red",   linestyle="-",  lw=1.5, label=f"Media = {media:.2f}")
    ax.axvline(mediana, color="orange",linestyle="--", lw=1.5, label=f"Mediana = {mediana:.2f}")
    ax.axvline(p5,      color="gray",  linestyle="--", lw=1, alpha=0.7, label=f"P5 = {p5:.2f}")
    ax.axvline(p95,     color="gray",  linestyle="--", lw=1, alpha=0.7, label=f"P95 = {p95:.2f}")
 
    ax.set_title(f"Distribución del precio simulado al cabo de "
                 f"{S.shape[1]-1} días  -  {nombre.upper()}")
    ax.set_xlabel("Precio final")
    ax.set_ylabel("Densidad")
    ax.legend(loc="best")
    ax.grid(alpha=0.3)
 
    ruta = os.path.join(carpeta, f"{nombre}_distribucion_final.png")
    fig.savefig(ruta)
    plt.close(fig)
    return ruta
 
 
def plot_historico_y_simulacion(
    df: pd.DataFrame, V: np.ndarray, nombre: str, carpeta: str,
    dias_historico: int = 500,
) -> str:
    """
    Gráfica conjunta: varianza histórica observada (parte izquierda)
    y trayectorias simuladas a partir del último valor (parte derecha).
    """
    var_hist = df["Var_Hist"].values[-dias_historico:]
 
    fig, ax = plt.subplots(figsize=(12, 5))
 
    # Histórico
    eje_hist = np.arange(-len(var_hist), 0)
    ax.plot(eje_hist, var_hist, color="black", lw=1.2,
            label="Varianza histórica")
 
    # Simulación
    n_mostrar = min(50, V.shape[0])
    eje_sim = np.arange(0, V.shape[1])
    for i in range(n_mostrar):
        ax.plot(eje_sim, V[i], lw=0.5, alpha=0.2, color="steelblue")
 
    media_sim = V.mean(axis=0)
    p5  = np.percentile(V, 5, axis=0)
    p95 = np.percentile(V, 95, axis=0)
    ax.plot(eje_sim, media_sim, color="navy", lw=2, label="Media simulada")
    ax.fill_between(eje_sim, p5, p95, color="navy", alpha=0.15,
                    label="IC 90% simulado")
 
    ax.axvline(0, color="red", linestyle="--", lw=1, alpha=0.7)
    ax.set_title(f"Varianza histórica y simulada  -  {nombre.upper()}")
    ax.set_xlabel("Días de trading (0 = hoy)")
    ax.set_ylabel("Varianza anualizada")
    ax.legend(loc="best")
    ax.grid(alpha=0.3)
 
    ruta = os.path.join(carpeta, f"{nombre}_historico_simulacion.png")
    fig.savefig(ruta)
    plt.close(fig)
    return ruta
 
 
def calcular_estadisticas(
    S: np.ndarray, V: np.ndarray, s0: float, nombre: str
) -> dict:
    """Calcula estadísticas resumen sobre las trayectorias."""
    precio_final = S[:, -1]
    var_final    = V[:, -1]
 
    return {
        "activo":          nombre.upper(),
        "S0":              s0,
        "precio_medio":    precio_final.mean(),
        "precio_mediano":  np.median(precio_final),
        "precio_p5":       np.percentile(precio_final, 5),
        "precio_p95":      np.percentile(precio_final, 95),
        "retorno_medio_%": (precio_final.mean() / s0 - 1) * 100,
        "vol_simulada":    np.sqrt(np.mean(var_final)),
        "var_media_final": var_final.mean(),
    }
 
 
# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================
 
def main():
    print("=" * 70)
    print("Simulación del modelo de Heston (parámetros: regresión lineal)")
    print("=" * 70)
    print(f"Horizonte:           {HORIZONTE} días")
    print(f"Trayectorias MC:     {N_SIMS}")
    print(f"Carpeta figuras:     {CARPETA_FIGURAS}/")
    print("-" * 70)
 
    os.makedirs(CARPETA_FIGURAS, exist_ok=True)
 
    # Cargar parámetros calibrados
    params_df = cargar_parametros(
        CARPETA_RESULTADOS, "parametros_regresion_lineal.csv"
    )
 
    estadisticas = []
 
    for nombre in ACTIVOS:
        print(f"\n>>> Simulando {nombre.upper()}...")
 
        df = cargar_datos(nombre, CARPETA_DATOS)
 
        # Parámetros de este activo
        fila = params_df[params_df["activo"] == nombre.upper()].iloc[0]
        kappa = float(fila["kappa"])
        theta = float(fila["theta"])
        xi    = float(fila["xi"])
        rho   = float(fila["rho"])
        v0    = float(fila["v0"])
 
        # Precio inicial: último precio observado
        col_precio = "Adj Close" if "Adj Close" in df.columns else "Close"
        s0 = float(df[col_precio].iloc[-1])
 
        # Drift mu: media anualizada de los retornos logarítmicos
        mu = float(df["Log_Return"].mean()) * DIAS_ANIO
 
        print(f"    S0={s0:.2f}, v0={v0:.4f}, mu={mu:.4f}")
        print(f"    kappa={kappa:.4f}, theta={theta:.4f}, xi={xi:.4f}, rho={rho:.4f}")
 
        # Simulación
        S, V = simular_heston(
            s0=s0, v0=v0, mu=mu,
            kappa=kappa, theta=theta, xi=xi, rho=rho,
            dt=DT, horizonte=HORIZONTE, n_sims=N_SIMS,
            semilla=SEMILLA,
        )
 
        # Gráficas
        plot_trayectorias_varianza(V, theta, nombre, CARPETA_FIGURAS)
        plot_trayectorias_precio(S, s0, nombre, CARPETA_FIGURAS)
        plot_distribucion_final(S, s0, nombre, CARPETA_FIGURAS)
        plot_historico_y_simulacion(df, V, nombre, CARPETA_FIGURAS)
 
        # Estadísticas
        stats = calcular_estadisticas(S, V, s0, nombre)
        estadisticas.append(stats)
 
        print(f"    Precio medio final: {stats['precio_medio']:.2f}")
        print(f"    Retorno medio:      {stats['retorno_medio_%']:.2f}%")
        print(f"    Vol simulada:       {stats['vol_simulada']*100:.2f}%")
 
    # Guardar estadísticas
    print("\n" + "-" * 70)
    ruta_stats = os.path.join(
        CARPETA_RESULTADOS, "estadisticas_simulacion_regresion_lineal.csv"
    )
    pd.DataFrame(estadisticas).to_csv(ruta_stats, index=False)
    print(f"Estadísticas guardadas en: {ruta_stats}")
    print(f"Gráficas guardadas en:     {CARPETA_FIGURAS}/")
    print("=" * 70)
 
 
if __name__ == "__main__":
    main()
 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression


# ============================================================
# CONFIGURACIÓN
# ============================================================

ACTIVOS = ["sp500", "aapl", "msft", "gold"]

CARPETA_DATOS      = "datos"
CARPETA_FIGURAS    = "figuras/regresion_lineal"
CARPETA_RESULTADOS = "resultados"

DT        = 1 / 252
DIAS_ANIO = 252
N_SIMS    = 10000
SEMILLA   = 42

# Bloque A: fechas de corte para backtesting (mayo, junio, julio 2025)
FECHAS_CORTE = ["2025-05-01", "2025-06-01", "2025-07-01"]

# Bloque B: horizontes de predicción a futuro
HORIZONTES_FUTURO = {
    "1 mes":   21,
    "3 meses": 63,
    "6 meses": 126,
    "1 año":   252,
}

# Ventana de zoom (en días de trading) para la gráfica ampliada
DIAS_ZOOM = 126  # último semestre del histórico para el panel ampliado

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})


# ============================================================
# CARGA DE DATOS
# ============================================================

def cargar_datos(nombre: str) -> pd.DataFrame:
    """Carga el CSV de un activo y devuelve un DataFrame limpio."""
    ruta = os.path.join(CARPETA_DATOS, f"{nombre}.csv")
    df = pd.read_csv(ruta, index_col=0, parse_dates=True)
    df = df.dropna(subset=["Log_Return", "Var_Hist"])
    return df


# ============================================================
# CALIBRACIÓN POR REGRESIÓN LINEAL
# ============================================================

def calibrar_regresion_lineal(df: pd.DataFrame, dt: float = DT) -> dict:
    """
    Calibra los seis parámetros (kappa, theta, xi, rho, v0, mu) del
    modelo de Heston mediante regresión lineal sobre los incrementos
    de la varianza, usando SOLO los datos del DataFrame que se le pasa.
    Cada llamada recalcula todos los parámetros incluido mu.
    """
    var_hist   = df["Var_Hist"].values
    log_return = df["Log_Return"].values

    v_t     = var_hist[:-1]
    delta_v = var_hist[1:] - v_t

    X = v_t.reshape(-1, 1)
    Y = delta_v
    modelo = LinearRegression().fit(X, Y)

    intercept = modelo.intercept_
    pendiente = modelo.coef_[0]

    kappa = max(-pendiente / dt, 1e-4)
    theta = max(intercept / (kappa * dt), 1e-4)

    residuos = delta_v - modelo.predict(X)
    xi = np.std(residuos) / np.sqrt(np.mean(v_t) * dt)

    log_ret_alineado = log_return[-len(residuos):]
    rho = np.corrcoef(log_ret_alineado, residuos)[0, 1]
    if np.isnan(rho):
        rho = 0.0

    v0 = var_hist[-1]
    # mu: media anualizada de los retornos logarítmicos del subconjunto
    mu = float(np.mean(log_return)) * DIAS_ANIO

    return {
        "kappa": kappa, "theta": theta, "xi": xi,
        "rho": rho, "v0": v0, "mu": mu,
    }


# ============================================================
# SIMULACIÓN DE HESTON (Euler-Maruyama)
# ============================================================

def simular_heston(
    s0, v0, mu, kappa, theta, xi, rho,
    horizonte, n_sims=N_SIMS, dt=DT, semilla=SEMILLA,
):
    """
    Simula n_sims trayectorias del modelo de Heston durante
    'horizonte' pasos temporales. Primero se resuelve la ecuación
    de la varianza (Euler-Maruyama con truncamiento) y luego la
    del precio usando esa varianza.
    """
    rng = np.random.default_rng(semilla)

    S = np.zeros((n_sims, horizonte + 1))
    V = np.zeros((n_sims, horizonte + 1))
    S[:, 0] = s0
    V[:, 0] = v0

    Z1 = rng.standard_normal((n_sims, horizonte))
    Z2_indep = rng.standard_normal((n_sims, horizonte))
    rho = np.clip(rho, -0.999, 0.999)
    Z2 = rho * Z1 + np.sqrt(1 - rho**2) * Z2_indep

    sqrt_dt = np.sqrt(dt)

    for t in range(horizonte):
        v_prev = np.maximum(V[:, t], 0.0)
        sqrt_v = np.sqrt(v_prev)

        V[:, t+1] = np.maximum(
            v_prev + kappa * (theta - v_prev) * dt
            + xi * sqrt_v * sqrt_dt * Z1[:, t],
            0.0,
        )
        S[:, t+1] = S[:, t] * np.exp(
            (mu - 0.5 * v_prev) * dt + sqrt_v * sqrt_dt * Z2[:, t]
        )

    return S, V


def fechas_futuras(fecha_inicio: pd.Timestamp, horizonte: int) -> pd.DatetimeIndex:
    """Genera un índice de fechas de trading desde fecha_inicio."""
    return pd.bdate_range(start=fecha_inicio, periods=horizonte + 1)


def _formato_fechas(ax):
    """Aplica formato de fechas reales (años) al eje X."""
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))


def _formato_fechas_meses(ax):
    """Aplica formato de fechas reales (meses) al eje X para gráficas ampliadas."""
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")


# ============================================================
# BLOQUE A — BACKTESTING HISTÓRICO
# ============================================================

def backtesting_historico(df_full: pd.DataFrame, nombre: str) -> plt.Figure:
    """
    Para un activo, hace tres cortes (mayo, junio, julio 2025), recalibra
    el modelo con los datos hasta cada corte y simula hasta hoy.
    Cada panel muestra TODO el histórico + simulación + precio real.
    """
    col_precio = "Adj Close" if "Adj Close" in df_full.columns else "Close"

    fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=False)
    fig.suptitle(
        f"Backtesting — {nombre.upper()}  (parámetros: regresión lineal)",
        fontsize=14, fontweight="bold",
    )

    for ax, fecha_corte in zip(axes, FECHAS_CORTE):
        fecha_corte_ts = pd.Timestamp(fecha_corte)

        df_train = df_full.loc[df_full.index < fecha_corte_ts]
        df_test  = df_full.loc[df_full.index >= fecha_corte_ts]

        if len(df_train) < 100 or len(df_test) < 5:
            ax.set_title(f"Corte {fecha_corte} (datos insuficientes)")
            continue

        # 1) Recalibrar TODOS los parámetros (incluido mu) con datos hasta el corte
        params = calibrar_regresion_lineal(df_train)
        s0 = float(df_train[col_precio].iloc[-1])
        horizonte = len(df_test)

        # 2) Simular desde el corte hacia adelante
        S, V = simular_heston(
            s0=s0, v0=params["v0"], mu=params["mu"],
            kappa=params["kappa"], theta=params["theta"],
            xi=params["xi"], rho=params["rho"],
            horizonte=horizonte,
        )

        # 3) Fechas reales
        fechas_hist = df_train.index
        fechas_sim  = fechas_futuras(fecha_corte_ts, horizonte)
        precio_real = df_test[col_precio].values
        fechas_real = df_test.index

        # 4) Histórico completo
        ax.plot(fechas_hist, df_train[col_precio].values,
                color="black", lw=1.2, label="Histórico (datos reales)")

        # 5) Trayectorias simuladas
        for i in range(min(150, N_SIMS)):
            ax.plot(fechas_sim, S[i], lw=0.4, alpha=0.10, color="steelblue")

        media = S.mean(axis=0)
        p5  = np.percentile(S, 5, axis=0)
        p95 = np.percentile(S, 95, axis=0)

        ax.plot(fechas_sim, media, color="navy", lw=2, label="Media simulada")
        ax.fill_between(fechas_sim, p5, p95, color="navy", alpha=0.15, label="IC 90%")

        # 6) Precio real desde el corte
        ax.plot(fechas_real, precio_real,
                color="red", lw=2.2, label="Precio real (test)")

        # 7) Línea vertical en el corte
        ax.axvline(fecha_corte_ts, color="gray", linestyle="--", lw=1.5, alpha=0.7)

        # 8) Métricas
        precio_final_real = precio_real[-1]
        idx_final = min(len(precio_real) - 1, len(media) - 1)
        precio_final_sim  = media[idx_final]
        error_pct = abs(precio_final_sim - precio_final_real) / precio_final_real * 100

        ax.set_title(
            f"Corte: {fecha_corte}   ({horizonte} días de predicción)   "
            f"S₀={s0:.2f} | Real={precio_final_real:.2f} | "
            f"Sim={precio_final_sim:.2f} | Err={error_pct:.1f}%",
            fontsize=10,
        )
        ax.set_xlabel("Fecha")
        ax.set_ylabel("Precio")
        ax.legend(loc="best", fontsize=9)
        ax.grid(alpha=0.3)
        _formato_fechas(ax)

    fig.tight_layout()
    return fig


# ============================================================
# BLOQUE B — PREDICCIÓN A FUTURO (vista completa + ampliada)
# ============================================================

def prediccion_futura(df_full: pd.DataFrame, nombre: str) -> plt.Figure:
    """
    Calibra con todos los datos disponibles y simula a 4 horizontes.
    Cada horizonte tiene DOS paneles: uno con todo el histórico y otro
    ampliado a los últimos meses para ver mejor el detalle.
    """
    col_precio = "Adj Close" if "Adj Close" in df_full.columns else "Close"

    # Recalibrar TODOS los parámetros con todo el histórico
    params = calibrar_regresion_lineal(df_full)
    s0 = float(df_full[col_precio].iloc[-1])
    fecha_hoy = df_full.index[-1]

    n_horizontes = len(HORIZONTES_FUTURO)

    # Una fila por horizonte, dos columnas: vista completa + ampliada
    fig, axes = plt.subplots(n_horizontes, 2, figsize=(20, 5 * n_horizontes))

    fig.suptitle(
        f"Predicción a futuro — {nombre.upper()}  "
        f"(parámetros: regresión lineal)\n"
        f"Fecha base: {fecha_hoy.strftime('%Y-%m-%d')}   |   "
        f"S₀ = {s0:.2f}",
        fontsize=13, fontweight="bold",
    )

    for fila, (titulo, horizonte) in enumerate(HORIZONTES_FUTURO.items()):
        S, V = simular_heston(
            s0=s0, v0=params["v0"], mu=params["mu"],
            kappa=params["kappa"], theta=params["theta"],
            xi=params["xi"], rho=params["rho"],
            horizonte=horizonte,
        )

        fechas_hist = df_full.index
        fechas_sim  = fechas_futuras(fecha_hoy, horizonte)

        media = S.mean(axis=0)
        p5  = np.percentile(S, 5, axis=0)
        p95 = np.percentile(S, 95, axis=0)

        # ----- Panel IZQUIERDO: vista completa (todo el histórico) -----
        ax_full = axes[fila, 0]

        ax_full.plot(fechas_hist, df_full[col_precio].values,
                     color="black", lw=1.2, label="Histórico")
        for i in range(min(150, N_SIMS)):
            ax_full.plot(fechas_sim, S[i], lw=0.4, alpha=0.10, color="forestgreen")
        ax_full.plot(fechas_sim, media, color="darkgreen", lw=2, label="Media simulada")
        ax_full.fill_between(fechas_sim, p5, p95, color="darkgreen", alpha=0.15, label="IC 90%")
        ax_full.axvline(fecha_hoy, color="gray", linestyle="--", lw=1.5, alpha=0.7)

        ax_full.set_title(
            f"{titulo}   |   Vista completa   |   "
            f"Media final: {media[-1]:.2f}   |   "
            f"P5={p5[-1]:.2f}  P95={p95[-1]:.2f}",
            fontsize=10,
        )
        ax_full.set_xlabel("Fecha")
        ax_full.set_ylabel("Precio")
        ax_full.legend(loc="best", fontsize=8)
        ax_full.grid(alpha=0.3)
        _formato_fechas(ax_full)

        # ----- Panel DERECHO: vista ampliada (últimos meses + predicción) -----
        ax_zoom = axes[fila, 1]

        fechas_hist_zoom = fechas_hist[-DIAS_ZOOM:]
        precio_hist_zoom = df_full[col_precio].values[-DIAS_ZOOM:]

        ax_zoom.plot(fechas_hist_zoom, precio_hist_zoom,
                     color="black", lw=1.5, label="Histórico (últimos 6 meses)")
        for i in range(min(150, N_SIMS)):
            ax_zoom.plot(fechas_sim, S[i], lw=0.4, alpha=0.12, color="forestgreen")
        ax_zoom.plot(fechas_sim, media, color="darkgreen", lw=2, label="Media simulada")
        ax_zoom.fill_between(fechas_sim, p5, p95, color="darkgreen", alpha=0.15, label="IC 90%")
        ax_zoom.axvline(fecha_hoy, color="gray", linestyle="--", lw=1.5, alpha=0.7)

        ax_zoom.set_title(
            f"{titulo}   |   Vista ampliada (últimos 6 meses + predicción)",
            fontsize=10,
        )
        ax_zoom.set_xlabel("Fecha")
        ax_zoom.set_ylabel("Precio")
        ax_zoom.legend(loc="best", fontsize=8)
        ax_zoom.grid(alpha=0.3)
        _formato_fechas_meses(ax_zoom)

    fig.tight_layout()
    return fig


# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    print("=" * 72)
    print(" SIMULACIÓN DE HESTON — PARÁMETROS POR REGRESIÓN LINEAL ")
    print("=" * 72)
    print(f"Activos:           {ACTIVOS}")
    print(f"Trayectorias MC:   {N_SIMS}")
    print(f"Fechas de corte:   {FECHAS_CORTE}")
    print(f"Horizontes futuro: {list(HORIZONTES_FUTURO.keys())}")
    print(f"Carpeta figuras:   {CARPETA_FIGURAS}/")
    print("-" * 72)

    os.makedirs(CARPETA_FIGURAS, exist_ok=True)

    for nombre in ACTIVOS:
        print(f"\n>>> {nombre.upper()}")
        df = cargar_datos(nombre)
        print(f"    Rango de datos: "
              f"{df.index[0].strftime('%Y-%m-%d')} → "
              f"{df.index[-1].strftime('%Y-%m-%d')}   "
              f"({len(df)} filas)")

        # Bloque A: Backtesting
        print("    [Bloque A] Backtesting histórico (3 cortes)...")
        fig_A = backtesting_historico(df, nombre)
        ruta_A = os.path.join(CARPETA_FIGURAS, f"{nombre}_A_backtesting.png")
        fig_A.savefig(ruta_A)

        # Bloque B: Predicción futura
        print("    [Bloque B] Predicción a futuro (4 horizontes)...")
        fig_B = prediccion_futura(df, nombre)
        ruta_B = os.path.join(CARPETA_FIGURAS, f"{nombre}_B_futuro.png")
        fig_B.savefig(ruta_B)

    print("\n" + "-" * 72)
    print(f"Guardadas {len(ACTIVOS)*2} figuras en: {CARPETA_FIGURAS}/")
    print("Mostrando todas las gráficas por pantalla...")
    print("=" * 72)

    plt.show()


if __name__ == "__main__":
    main()


## red neurnoal

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# CONFIGURACIÓN
# ============================================================

ACTIVOS = ["sp500", "aapl", "msft", "gold"]

CARPETA_DATOS      = "datos"
CARPETA_FIGURAS    = "figuras/red_neuronal"
CARPETA_RESULTADOS = "resultados"

DT        = 1 / 252
DIAS_ANIO = 252
N_SIMS    = 10000
SEMILLA   = 42

# Hiperparámetros de la calibración con Adam
LEARNING_RATE = 0.01
N_ITER        = 10000
TOL           = 1e-9

# Bloque A: fechas de corte para backtesting (mayo, junio, julio 2025)
FECHAS_CORTE = ["2025-05-01", "2025-06-01", "2025-07-01"]

# Bloque B: horizontes de predicción a futuro
HORIZONTES_FUTURO = {
    "1 mes":   21,
    "3 meses": 63,
    "6 meses": 126,
    "1 año":   252,
}

DIAS_ZOOM = 126  # último semestre para los paneles ampliados

torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})


# ============================================================
# CARGA DE DATOS
# ============================================================

def cargar_datos(nombre: str) -> pd.DataFrame:
    ruta = os.path.join(CARPETA_DATOS, f"{nombre}.csv")
    df = pd.read_csv(ruta, index_col=0, parse_dates=True)
    df = df.dropna(subset=["Log_Return", "Var_Hist"])
    return df


# ============================================================
# CALIBRACIÓN POR RED NEURONAL
# ------------------------------------------------------------
# Se entrena UNA red neuronal (perceptrón multicapa) que aprende el
# mapa inverso del modelo: a partir de las características de una
# trayectoria de varianza, predice los parámetros (kappa, theta, xi).
# El nivel de largo plazo theta se ancla a la media observada de la
# serie (theta = media * exp(corrección acotada)), de modo que los
# picos de la serie afectan a la forma pero no al nivel. La red se
# entrena una sola vez con datos sintéticos del proceso CIR y luego se
# aplica, de forma instantánea, a cada serie real y a cada subserie
# del backtesting.
# ============================================================

# Hiperparámetros de la red
N_MUESTRAS_ENTRENO = 6000     # trayectorias sintéticas de entrenamiento
LONGITUD_SERIE     = 2500     # longitud de cada trayectoria (~10 años)
N_EPOCAS           = 3000
LR_RED             = 1e-3
N_OCULTAS          = 128      # neuronas por capa oculta
MARGEN_THETA       = 0.25     # margen del nivel de largo plazo en torno a la media


def simular_cir_batch(kappa, theta, xi, v0, n, dt=DT, rng=None):
    """Simula len(kappa) trayectorias CIR en paralelo. Devuelve (N, n)."""
    if rng is None:
        rng = np.random.default_rng(0)
    N = len(kappa)
    v = np.empty((N, n))
    v[:, 0] = v0
    sq_dt = np.sqrt(dt)
    for t in range(1, n):
        prev = v[:, t - 1]
        z = rng.standard_normal(N)
        dv = kappa * (theta - prev) * dt + xi * np.sqrt(np.maximum(prev, 0)) * sq_dt * z
        v[:, t] = np.maximum(prev + dv, 1e-8)
    return v


def autocorr_batch(V, lag):
    a, b = V[:, :-lag], V[:, lag:]
    am, bm = a.mean(1, keepdims=True), b.mean(1, keepdims=True)
    num = ((a - am) * (b - bm)).sum(1)
    den = np.sqrt(((a - am) ** 2).sum(1) * ((b - bm) ** 2).sum(1)) + 1e-12
    return num / den


def features_batch(V):
    """Devuelve (caracteristicas, log_media). El nivel se separa de la forma."""
    V = V.astype(np.float64)
    dv = np.diff(V, axis=1)
    v_t = V[:, :-1]
    n = v_t.shape[1]
    m = V.mean(1)
    sx, sy = v_t.sum(1), dv.sum(1)
    sxx, sxy = (v_t ** 2).sum(1), (v_t * dv).sum(1)
    pend = (n * sxy - sx * sy) / (n * sxx - sx ** 2 + 1e-18)
    corte = (sy - pend * sx) / n
    resid = dv - (pend[:, None] * v_t + corte[:, None])
    log_m = np.log(m)
    X = np.column_stack([
        log_m, pend, resid.std(1) / np.sqrt(m),
        V.std(1) / m, dv.std(1) / m,
        autocorr_batch(V, 1), autocorr_batch(V, 5),
        autocorr_batch(V, 10), autocorr_batch(V, 20),
    ])
    return X, log_m


def features_serie(v):
    X, log_m = features_batch(v.reshape(1, -1))
    return X[0], log_m[0]


def generar_dataset(n_muestras, n=LONGITUD_SERIE, rng=None):
    """Muestrea parámetros, simula trayectorias CIR y extrae características."""
    if rng is None:
        rng = np.random.default_rng(0)
    kappa = np.exp(rng.uniform(np.log(0.5),   np.log(8.0),  n_muestras))
    theta = np.exp(rng.uniform(np.log(0.005), np.log(0.20), n_muestras))
    xi    = np.exp(rng.uniform(np.log(0.2),   np.log(1.5),  n_muestras))
    V = simular_cir_batch(kappa, theta, xi, theta, n, rng=rng)
    X, log_m = features_batch(V)
    correccion = np.clip(np.log(theta) - log_m, -MARGEN_THETA, MARGEN_THETA)
    Y = np.column_stack([correccion, np.log(kappa), np.log(xi)])
    return X, Y


class RedCalibradora(nn.Module):
    """
    Perceptrón multicapa (MLP) feed-forward.
    Arquitectura: 9 -> 128 -> 128 -> 128 -> 3, con activación ReLU.
    Salida: (corrección de theta acotada, log kappa, log xi).
    """
    def __init__(self, n_in=9, h=N_OCULTAS, n_out=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, h), nn.ReLU(),
            nn.Linear(h, h),    nn.ReLU(),
            nn.Linear(h, h),    nn.ReLU(),
            nn.Linear(h, n_out),
        )

    def forward(self, x):
        o = self.net(x)
        return torch.cat([MARGEN_THETA * torch.tanh(o[:, :1]), o[:, 1:]], dim=1)


# Variables globales: la red entrenada y la normalización de las características
_RED = None
_X_MEAN = None
_X_STD = None


def entrenar_red_global():
    """Entrena la red una sola vez y guarda la red y la normalización."""
    global _RED, _X_MEAN, _X_STD
    torch.manual_seed(SEMILLA)
    np.random.seed(SEMILLA)

    print("Entrenando la red neuronal (una sola vez)...")
    X_train, Y_train = generar_dataset(N_MUESTRAS_ENTRENO,
                                        rng=np.random.default_rng(SEMILLA))
    _X_MEAN, _X_STD = X_train.mean(0), X_train.std(0)

    Xn = np.clip((X_train - _X_MEAN) / _X_STD, -5, 5)
    Xt = torch.tensor(Xn, dtype=torch.float64)
    Yt = torch.tensor(Y_train, dtype=torch.float64)

    red = RedCalibradora().double()
    opt = optim.Adam(red.parameters(), lr=LR_RED)
    lossf = nn.MSELoss()
    for epoch in range(N_EPOCAS):
        opt.zero_grad()
        loss = lossf(red(Xt), Yt)
        loss.backward()
        opt.step()
        if epoch % 1000 == 0:
            print(f"  época {epoch:5d}   pérdida {loss.item():.5f}")
    print(f"  pérdida final: {loss.item():.5f}")
    _RED = red


def calibrar_red_neuronal(df: pd.DataFrame, dt: float = DT) -> dict:
    """
    Calibra los parámetros de Heston aplicando la red neuronal a las
    características de la serie de varianza del DataFrame recibido.
    La red estima theta (anclado a la media), kappa y xi. Los parámetros
    rho, v0 y mu se calculan directamente sobre los datos.
    """
    if _RED is None:
        entrenar_red_global()

    var_hist   = df["Var_Hist"].values.astype(np.float64)
    log_return = df["Log_Return"].values.astype(np.float64)

    # --- La red estima theta, kappa, xi ---
    fvec, log_m = features_serie(var_hist)
    f = np.clip((fvec - _X_MEAN) / _X_STD, -5, 5)
    with torch.no_grad():
        o = _RED(torch.tensor(f, dtype=torch.float64).reshape(1, -1)).numpy()[0]
    theta = float(np.exp(o[0] + log_m))
    kappa = float(np.clip(np.exp(o[1]), 0.5, 8.0))  # se restringe al rango de entrenamiento de la red (extrapolar fuera no tiene soporte)
    xi    = float(np.exp(o[2]))

    # --- rho: correlación entre retornos y residuos del drift CIR ---
    v_t = var_hist[:-1]
    delta_v = var_hist[1:] - v_t
    residuos_final = delta_v - kappa * (theta - v_t) * dt
    log_ret_alineado = log_return[-len(residuos_final):]
    rho = np.corrcoef(log_ret_alineado, residuos_final)[0, 1]
    if np.isnan(rho):
        rho = 0.0
    rho = float(np.clip(rho, -0.999, 0.999))

    # --- v0: última varianza observada ---
    v0 = float(var_hist[-1])

    # --- mu: media anualizada de los retornos ---
    mu = float(np.mean(log_return)) * DIAS_ANIO

    # --- R2 diagnóstico del ajuste de la deriva ---
    ss_res = np.sum(residuos_final ** 2)
    ss_tot = np.sum((delta_v - np.mean(delta_v)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else np.nan

    return {
        "kappa": kappa, "theta": theta, "xi": xi,
        "rho": rho, "v0": v0, "mu": mu, "R2": r2,
    }


# ============================================================
# SIMULACIÓN DE HESTON (igual que regresión lineal)
# ============================================================

def simular_heston(
    s0, v0, mu, kappa, theta, xi, rho,
    horizonte, n_sims=N_SIMS, dt=DT, semilla=SEMILLA,
):
    """
    Resuelve primero la ecuación de la varianza (2.5) por Euler-Maruyama,
    y luego la ecuación del precio (2.4) usando esa varianza.
    """
    rng = np.random.default_rng(semilla)

    S = np.zeros((n_sims, horizonte + 1))
    V = np.zeros((n_sims, horizonte + 1))
    S[:, 0] = s0
    V[:, 0] = v0

    Z1 = rng.standard_normal((n_sims, horizonte))
    Z2_indep = rng.standard_normal((n_sims, horizonte))
    rho = np.clip(rho, -0.999, 0.999)
    Z2 = rho * Z1 + np.sqrt(1 - rho**2) * Z2_indep

    sqrt_dt = np.sqrt(dt)

    for t in range(horizonte):
        v_prev = np.maximum(V[:, t], 0.0)
        sqrt_v = np.sqrt(v_prev)

        # Ecuación 2.5: dv_t
        V[:, t + 1] = np.maximum(
            v_prev + kappa * (theta - v_prev) * dt
            + xi * sqrt_v * sqrt_dt * Z1[:, t],
            0.0
        )
        # Ecuación 2.4: dS_t
        S[:, t + 1] = S[:, t] * np.exp(
            (mu - 0.5 * v_prev) * dt + sqrt_v * sqrt_dt * Z2[:, t]
        )

    return S, V


def fechas_futuras(fecha_inicio: pd.Timestamp, horizonte: int) -> pd.DatetimeIndex:
    return pd.bdate_range(start=fecha_inicio, periods=horizonte + 1)


def _formato_fechas(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))


def _formato_fechas_meses(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")


# ============================================================
# BLOQUE A — BACKTESTING HISTÓRICO
# ============================================================

def dibujar_tendencias(ax, df_train, fechas_sim, col_precio):
    """
    Ajusta rectas de tendencia sobre el histórico de entrenamiento en tres
    ventanas (últimos 6 meses, último año y todo el histórico) y las proyecta
    hasta el final de la simulación. Cada recta indica su pendiente anualizada
    en porcentaje, calculada respecto al precio medio de su ventana.
    """
    precios = df_train[col_precio].values.astype(float)
    fechas_hist = df_train.index
    n_hist = len(precios)

    # Fechas combinadas (histórico + simulación) para proyectar la recta
    fechas_ext = fechas_hist.append(pd.DatetimeIndex(fechas_sim))
    n_ext = len(fechas_ext)

    ventanas = [
        ("Últimos 6 meses", 126,  "tab:red"),
        ("Último año",      252,  "tab:orange"),
        ("Todo el histórico", None, "tab:cyan"),
    ]
    for etiqueta, dias, color in ventanas:
        if dias is None:
            ini = 0
        else:
            if n_hist <= dias:
                continue
            ini = n_hist - dias

        # Ajuste lineal sobre la ventana
        x_fit = np.arange(n_hist - ini)
        y_fit = precios[ini:]
        a, b = np.polyfit(x_fit, y_fit, 1)
        pend_anual = a * DIAS_ANIO / np.mean(y_fit) * 100.0

        # Proyección de la recta desde el inicio de la ventana hasta el final
        x_plot = np.arange(n_ext - ini)
        y_plot = a * x_plot + b
        ax.plot(fechas_ext[ini:], y_plot, linestyle="--", lw=1.6,
                color=color, alpha=0.9,
                label=f"{etiqueta} ({pend_anual:+.1f}%/año)")


def backtesting_historico_red_neuronal(df_full: pd.DataFrame, nombre: str) -> plt.Figure:
    """
    Backtesting con red neuronal: 3 cortes (mayo, junio, julio 2025).
    Recalibra TODOS los parámetros en cada corte y simula hasta hoy.
    """
    col_precio = "Adj Close" if "Adj Close" in df_full.columns else "Close"

    fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=False)
    fig.suptitle(
        f"Backtesting — {nombre.upper()}  (parámetros: red neuronal)",
        fontsize=14, fontweight="bold",
    )

    for ax, fecha_corte in zip(axes, FECHAS_CORTE):
        fecha_corte_ts = pd.Timestamp(fecha_corte)

        df_train = df_full.loc[df_full.index < fecha_corte_ts]
        df_test  = df_full.loc[df_full.index >= fecha_corte_ts]

        if len(df_train) < 100 or len(df_test) < 5:
            ax.set_title(f"Corte {fecha_corte} (datos insuficientes)")
            continue

        # Recalibrar con red neuronal usando solo datos hasta el corte
        params = calibrar_red_neuronal(df_train)

        s0 = float(df_train[col_precio].iloc[-1])
        horizonte = len(df_test)

        S, V = simular_heston(
            s0=s0, v0=params["v0"], mu=params["mu"],
            kappa=params["kappa"], theta=params["theta"],
            xi=params["xi"], rho=params["rho"],
            horizonte=horizonte,
        )

        fechas_hist = df_train.index
        fechas_sim  = fechas_futuras(fecha_corte_ts, horizonte)
        precio_real = df_test[col_precio].values
        fechas_real = df_test.index

        # Histórico
        ax.plot(fechas_hist, df_train[col_precio].values,
                color="black", lw=1.2, label="Histórico (datos reales)")

        # Rectas de tendencia (6 meses, 1 año, todo el histórico)
        dibujar_tendencias(ax, df_train, fechas_sim, col_precio)

        # Trayectorias simuladas
        for i in range(min(150, N_SIMS)):
            ax.plot(fechas_sim, S[i], lw=0.4, alpha=0.10, color="darkorange")

        media = S.mean(axis=0)
        p5  = np.percentile(S, 5, axis=0)
        p95 = np.percentile(S, 95, axis=0)

        ax.plot(fechas_sim, media, color="saddlebrown", lw=2, label="Media simulada")
        ax.fill_between(fechas_sim, p5, p95, color="saddlebrown", alpha=0.15, label="IC 90%")

        # Precio real
        ax.plot(fechas_real, precio_real,
                color="red", lw=2.2, label="Precio real (test)")

        ax.axvline(fecha_corte_ts, color="gray", linestyle="--", lw=1.5, alpha=0.7)

        precio_final_real = precio_real[-1]
        idx_final = min(len(precio_real) - 1, len(media) - 1)
        precio_final_sim = media[idx_final]
        error_pct = abs(precio_final_sim - precio_final_real) / precio_final_real * 100

        ax.set_title(
            f"Corte: {fecha_corte}   ({horizonte} días de predicción)   "
            f"S₀={s0:.2f} | Real={precio_final_real:.2f} | "
            f"Sim={precio_final_sim:.2f} | Err={error_pct:.1f}%",
            fontsize=10,
        )
        ax.set_xlabel("Fecha")
        ax.set_ylabel("Precio")
        ax.legend(loc="best", fontsize=9)
        ax.grid(alpha=0.3)
        _formato_fechas(ax)

    fig.tight_layout()
    return fig


# ============================================================
# BLOQUE B — PREDICCIÓN A FUTURO (vista completa + ampliada)
# ============================================================

def prediccion_futura_red_neuronal(df_full: pd.DataFrame, nombre: str) -> plt.Figure:
    """
    Calibra con todos los datos mediante red neuronal y simula
    a 4 horizontes futuros. Cada horizonte muestra vista completa
    y vista ampliada (últimos 6 meses + predicción).
    """
    col_precio = "Adj Close" if "Adj Close" in df_full.columns else "Close"

    params = calibrar_red_neuronal(df_full)
    s0 = float(df_full[col_precio].iloc[-1])
    fecha_hoy = df_full.index[-1]

    n_horizontes = len(HORIZONTES_FUTURO)
    fig, axes = plt.subplots(n_horizontes, 2, figsize=(20, 5 * n_horizontes))

    fig.suptitle(
        f"Predicción a futuro — {nombre.upper()}  "
        f"(parámetros: red neuronal)\n"
        f"Fecha base: {fecha_hoy.strftime('%Y-%m-%d')}   |   "
        f"S₀ = {s0:.2f}",
        fontsize=13, fontweight="bold",
    )

    for fila, (titulo, horizonte) in enumerate(HORIZONTES_FUTURO.items()):
        S, V = simular_heston(
            s0=s0, v0=params["v0"], mu=params["mu"],
            kappa=params["kappa"], theta=params["theta"],
            xi=params["xi"], rho=params["rho"],
            horizonte=horizonte,
        )

        fechas_hist = df_full.index
        fechas_sim  = fechas_futuras(fecha_hoy, horizonte)

        media = S.mean(axis=0)
        p5  = np.percentile(S, 5, axis=0)
        p95 = np.percentile(S, 95, axis=0)

        # Vista completa
        ax_full = axes[fila, 0]
        ax_full.plot(fechas_hist, df_full[col_precio].values,
                     color="black", lw=1.2, label="Histórico")
        for i in range(min(150, N_SIMS)):
            ax_full.plot(fechas_sim, S[i], lw=0.4, alpha=0.10, color="purple")
        ax_full.plot(fechas_sim, media, color="indigo", lw=2, label="Media simulada")
        ax_full.fill_between(fechas_sim, p5, p95, color="indigo", alpha=0.15, label="IC 90%")
        ax_full.axvline(fecha_hoy, color="gray", linestyle="--", lw=1.5, alpha=0.7)
        ax_full.set_title(
            f"{titulo}   |   Vista completa   |   "
            f"Media final: {media[-1]:.2f}   |   "
            f"P5={p5[-1]:.2f}  P95={p95[-1]:.2f}",
            fontsize=10,
        )
        ax_full.set_xlabel("Fecha")
        ax_full.set_ylabel("Precio")
        ax_full.legend(loc="best", fontsize=8)
        ax_full.grid(alpha=0.3)
        _formato_fechas(ax_full)

        # Vista ampliada
        ax_zoom = axes[fila, 1]
        fechas_hist_zoom = fechas_hist[-DIAS_ZOOM:]
        precio_hist_zoom = df_full[col_precio].values[-DIAS_ZOOM:]

        ax_zoom.plot(fechas_hist_zoom, precio_hist_zoom,
                     color="black", lw=1.5, label="Histórico (últimos 6 meses)")
        for i in range(min(150, N_SIMS)):
            ax_zoom.plot(fechas_sim, S[i], lw=0.4, alpha=0.12, color="purple")
        ax_zoom.plot(fechas_sim, media, color="indigo", lw=2, label="Media simulada")
        ax_zoom.fill_between(fechas_sim, p5, p95, color="indigo", alpha=0.15, label="IC 90%")
        ax_zoom.axvline(fecha_hoy, color="gray", linestyle="--", lw=1.5, alpha=0.7)
        ax_zoom.set_title(
            f"{titulo}   |   Vista ampliada (últimos 6 meses + predicción)",
            fontsize=10,
        )
        ax_zoom.set_xlabel("Fecha")
        ax_zoom.set_ylabel("Precio")
        ax_zoom.legend(loc="best", fontsize=8)
        ax_zoom.grid(alpha=0.3)
        _formato_fechas_meses(ax_zoom)

    fig.tight_layout()
    return fig


# ============================================================
# GUARDAR PARÁMETROS
# ============================================================

def guardar_parametros_red_neuronal(resultados: list) -> str:
    os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
    ruta = os.path.join(CARPETA_RESULTADOS, "parametros_red_neuronal_simulacion.csv")
    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(ruta, index=False)
    return ruta


# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    print("=" * 72)
    print(" SIMULACIÓN DE HESTON — PARÁMETROS POR RED NEURONAL ")
    print("=" * 72)
    print(f"Activos:           {ACTIVOS}")
    print(f"Trayectorias MC:   {N_SIMS}")
    print(f"Fechas de corte:   {FECHAS_CORTE}")
    print(f"Horizontes futuro: {list(HORIZONTES_FUTURO.keys())}")
    print(f"Carpeta figuras:   {CARPETA_FIGURAS}/")
    print(f"Red:               MLP 10->128->128->128->3 (ReLU), Adam lr={LR_RED}")
    print("-" * 72)

    os.makedirs(CARPETA_FIGURAS, exist_ok=True)

    # Entrenar la red neuronal una sola vez (común a todos los activos y cortes)
    entrenar_red_global()
    print("-" * 72)

    resultados_parametros = []

    for nombre in ACTIVOS:
        print(f"\n>>> {nombre.upper()}")
        df = cargar_datos(nombre)
        print(f"    Rango: "
              f"{df.index[0].strftime('%Y-%m-%d')} → "
              f"{df.index[-1].strftime('%Y-%m-%d')}   "
              f"({len(df)} filas)")

        print("    Calibrando parámetros con toda la muestra...")
        params_full = calibrar_red_neuronal(df)

        resultados_parametros.append({"activo": nombre.upper(), **params_full})

        print(
            f"    kappa={params_full['kappa']:.4f} | "
            f"theta={params_full['theta']:.6f} | "
            f"xi={params_full['xi']:.4f} | "
            f"rho={params_full['rho']:.4f} | "
            f"v0={params_full['v0']:.6f} | "
            f"mu={params_full['mu']:.4f}"
        )

        # Bloque A
        print("    [Bloque A] Backtesting histórico (3 cortes)...")
        fig_A = backtesting_historico_red_neuronal(df, nombre)
        ruta_A = os.path.join(CARPETA_FIGURAS, f"{nombre}_A_backtesting.png")
        fig_A.savefig(ruta_A)

        # Bloque B
        print("    [Bloque B] Predicción a futuro (4 horizontes)...")
        fig_B = prediccion_futura_red_neuronal(df, nombre)
        ruta_B = os.path.join(CARPETA_FIGURAS, f"{nombre}_B_futuro.png")
        fig_B.savefig(ruta_B)

    ruta_params = guardar_parametros_red_neuronal(resultados_parametros)

    print("\n" + "-" * 72)
    print(f"Guardadas {len(ACTIVOS)*2} figuras en: {CARPETA_FIGURAS}/")
    print(f"Parámetros guardados en: {ruta_params}")
    print("Mostrando todas las gráficas por pantalla...")
    print("=" * 72)

    plt.show()


if __name__ == "__main__":
    main()


## Comparación visual entre regresión lineal y red neuronal

Esta sección genera figuras combinadas que muestran lado a lado las gráficas equivalentes de ambos métodos para cada activo. Hay que ejecutar antes las celdas de simulación de regresión lineal y de red neuronal, porque esta celda lee las imágenes ya guardadas en `figuras/regresion_lineal/` y `figuras/red_neuronal/`.

In [ ]:
# ============================================================
#  COMPARACIÓN VISUAL ENTRE REGRESIÓN LINEAL Y RED NEURONAL
# ============================================================
#
#  Para cada activo se generan figuras combinadas en las que se
#  muestran lado a lado las gráficas equivalentes de ambos métodos:
#  - Backtesting: regresión lineal vs red neuronal
#  - Predicción a futuro: regresión lineal vs red neuronal
#
#  Las imágenes individuales tienen que estar ya generadas en las
#  carpetas figuras/regresion_lineal/ y figuras/red_neuronal/.
# ============================================================

import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


ACTIVOS = ["sp500", "aapl", "msft", "gold"]
CARPETA_RL = "figuras/regresion_lineal"
CARPETA_RN = "figuras/red_neuronal"
CARPETA_COMP = "figuras/comparacion"

TIPOS_GRAFICA = {
    "A_backtesting": "Backtesting histórico",
    "B_futuro":      "Predicción a futuro",
}


def comparar_metodos(nombre: str, tipo: str, titulo: str) -> plt.Figure:
    """
    Pone lado a lado la figura del método de regresión lineal y
    la del método de red neuronal para un mismo activo y tipo de
    gráfica.
    """
    ruta_rl = os.path.join(CARPETA_RL, f"{nombre}_{tipo}.png")
    ruta_rn = os.path.join(CARPETA_RN, f"{nombre}_{tipo}.png")

    if not os.path.exists(ruta_rl) or not os.path.exists(ruta_rn):
        print(f"    [!] Falta alguna imagen: {ruta_rl} o {ruta_rn}")
        return None

    img_rl = mpimg.imread(ruta_rl)
    img_rn = mpimg.imread(ruta_rn)

    # Ajuste de tamaño en función de las proporciones de la imagen
    h_rl, w_rl = img_rl.shape[:2]
    h_rn, w_rn = img_rn.shape[:2]
    h = max(h_rl, h_rn)
    ancho_total = (w_rl + w_rn) / 100
    alto_total  = h / 100

    fig, axes = plt.subplots(1, 2, figsize=(ancho_total, alto_total))
    fig.suptitle(
        f"{titulo} — {nombre.upper()}   |   Regresión lineal (izq) vs Red neuronal (der)",
        fontsize=14, fontweight="bold",
    )

    axes[0].imshow(img_rl)
    axes[0].set_title("Regresión lineal", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(img_rn)
    axes[1].set_title("Red neuronal", fontsize=11)
    axes[1].axis("off")

    fig.tight_layout()
    return fig


def main():
    print("=" * 72)
    print(" COMPARACIÓN VISUAL — REGRESIÓN LINEAL vs RED NEURONAL ")
    print("=" * 72)
    print(f"Activos:           {ACTIVOS}")
    print(f"Carpeta salida:    {CARPETA_COMP}/")
    print("-" * 72)

    os.makedirs(CARPETA_COMP, exist_ok=True)

    for nombre in ACTIVOS:
        print(f"\n>>> {nombre.upper()}")
        for tipo, titulo in TIPOS_GRAFICA.items():
            print(f"    Comparando: {titulo}...")
            fig = comparar_metodos(nombre, tipo, titulo)
            if fig is None:
                continue
            ruta = os.path.join(CARPETA_COMP, f"{nombre}_{tipo}_COMP.png")
            fig.savefig(ruta, dpi=120, bbox_inches="tight")

    print("\n" + "-" * 72)
    print(f"Figuras comparativas guardadas en: {CARPETA_COMP}/")
    print("Mostrando todas las comparaciones por pantalla...")
    print("=" * 72)

    plt.show()


if __name__ == "__main__":
    main()


## Análisis de tendencias lineales

Esta celda añade tres tendencias lineales sobre cada gráfica (ajustadas a los últimos 6 meses, al último año y a todo el histórico) y permite comparar visualmente cómo de dispar es la simulación de Heston frente a una simple proyección lineal. Es independiente del resto del notebook: solo lee los CSVs de `datos/` y genera todo desde cero.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# ANÁLISIS DE TENDENCIAS LINEALES
# ------------------------------------------------------------
# Este análisis NO depende del método de calibración: ajusta
# rectas de tendencia directamente sobre la serie de precios
# de cada activo, en tres ventanas temporales (últimos 6 meses,
# último año y todo el histórico), y compara las pendientes.
# ============================================================

ACTIVOS = ["sp500", "aapl", "msft", "gold"]

CARPETA_DATOS   = "datos"
CARPETA_FIGURAS = "figuras/tendencias"

DIAS_ANIO = 252

# Ventanas de análisis (en días de cotización)
VENTANAS = {
    "6 meses": 126,
    "1 año":   252,
    "Total":   None,   # None = toda la serie disponible
}

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})


def cargar_precios(nombre: str) -> pd.Series:
    """Carga la serie de precios de cierre del activo."""
    ruta = os.path.join(CARPETA_DATOS, f"{nombre}.csv")
    df = pd.read_csv(ruta, index_col=0, parse_dates=True)
    col_precio = "Adj Close" if "Adj Close" in df.columns else "Close"
    return df[col_precio].dropna()


def ajustar_tendencia(precios: pd.Series):
    """
    Ajusta una recta a la serie de precios mediante mínimos cuadrados.
    Devuelve la pendiente anualizada en porcentaje (respecto al precio
    medio de la ventana) y la propia recta ajustada para graficar.
    """
    y = precios.values.astype(float)
    n = len(y)
    t = np.arange(n)                      # tiempo en días de cotización
    # Recta y = a*t + b
    a, b = np.polyfit(t, y, 1)
    recta = a * t + b
    # Pendiente anualizada como % del precio medio de la ventana
    pendiente_anual = (a * DIAS_ANIO) / np.mean(y) * 100.0
    return pendiente_anual, recta


def main():
    print("=" * 64)
    print("Análisis de tendencias lineales sobre los precios")
    print("=" * 64)
    os.makedirs(CARPETA_FIGURAS, exist_ok=True)

    resultados = {ventana: [] for ventana in VENTANAS}

    # ----- Figuras individuales: precio + rectas de tendencia -----
    for nombre in ACTIVOS:
        precios = cargar_precios(nombre)

        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.plot(precios.index, precios.values, color="black",
                linewidth=1.0, label="Precio")

        for ventana, dias in VENTANAS.items():
            sub = precios if dias is None else precios.iloc[-dias:]
            pend, recta = ajustar_tendencia(sub)
            resultados[ventana].append(pend)
            ax.plot(sub.index, recta, linestyle="--", linewidth=1.6,
                    label=f"Tendencia {ventana} ({pend:+.1f}%/año)")

        ax.set_title(f"Tendencias lineales — {nombre.upper()}")
        ax.set_xlabel("Fecha")
        ax.set_ylabel("Precio")
        ax.legend()
        ax.grid(alpha=0.3)
        ruta = os.path.join(CARPETA_FIGURAS, f"{nombre}_tendencias.png")
        fig.savefig(ruta)
        plt.close(fig)

        pendientes = [resultados[v][-1] for v in VENTANAS]
        print(f"\n{nombre.upper():6s}  " +
              "  ".join(f"{v}: {p:+.1f}%/año" for v, p in zip(VENTANAS, pendientes)))

    # ----- Figura comparativa de pendientes (barras) -----
    etiquetas = [a.upper() for a in ACTIVOS]
    x = np.arange(len(ACTIVOS))
    ancho = 0.25

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, ventana in enumerate(VENTANAS):
        ax.bar(x + (i - 1) * ancho, resultados[ventana], ancho, label=ventana)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Comparación de pendientes anualizadas por activo y horizonte")
    ax.set_ylabel("Pendiente (%/año)")
    ax.set_xticks(x)
    ax.set_xticklabels(etiquetas)
    ax.legend(title="Ventana")
    ax.grid(axis="y", alpha=0.3)
    ruta = os.path.join(CARPETA_FIGURAS, "TEND_comparativo_pendientes.png")
    fig.savefig(ruta)
    plt.close(fig)

    print("-" * 64)
    print(f"Figuras guardadas en: {CARPETA_FIGURAS}/")
    print("=" * 64)


if __name__ == "__main__":
    main()


In [ ]:
# =====================================================================
#  CELDA FINAL — genera TODAS las figuras del Capítulo 5 en la carpeta "final/"
#  Requiere haber ejecutado antes la celda de la RED NEURONAL, de modo que
#  existan: cargar_datos, calibrar_red_neuronal, simular_heston, fechas_futuras
#  y las constantes ACTIVOS, FECHAS_CORTE, HORIZONTES_FUTURO, DIAS_ZOOM,
#  N_SIMS, DT, DIAS_ANIO, SEMILLA.
# =====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CARPETA_FINAL = "final"
os.makedirs(CARPETA_FINAL, exist_ok=True)

# ---- Colores (predicción con color propio, todo bien diferenciado) ----
C_HIST  = "black"       # histórico
C_PRED  = "#2166ac"     # azul  -> media simulada (la predicción)
C_BANDA = "#2166ac"     # banda IC 90% (mismo tono, translúcido)
C_REAL  = "#d6181b"     # rojo  -> precio real
C_T6M   = "#f08c00"     # naranja  -> tendencia 6 meses
C_T1A   = "#8e44ad"     # morado   -> tendencia 1 año
C_TTOT  = "#138d75"     # verde    -> tendencia histórica
ZOOM_BT = 252           # días de histórico visibles antes del corte (ampliada)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
                     "font.size": 10, "axes.titlesize": 11.5, "axes.labelsize": 10,
                     "legend.fontsize": 8})

def _col(df):
    return "Adj Close" if "Adj Close" in df.columns else "Close"

def _tendencia(sub):
    """Ajusta y=a·t+b a la sub-serie; devuelve (a, b, pendiente_anual_%)."""
    y = sub.values.astype(float)
    a, b = np.polyfit(np.arange(len(y)), y, 1)
    return a, b, a * DIAS_ANIO / np.mean(y) * 100.0

# =====================================================================
#  1) BACKTESTING — corte de mayo, un activo por figura, con tendencias
# =====================================================================
CORTE = FECHAS_CORTE[0]                       # "2025-05-01"
corte_ts = pd.Timestamp(CORTE)
for nombre in ACTIVOS:
    df  = cargar_datos(nombre); col = _col(df)
    dtr = df.loc[df.index <  corte_ts]
    dte = df.loc[df.index >= corte_ts]

    p = calibrar_red_neuronal(dtr)
    s0 = float(dtr[col].iloc[-1]); H = len(dte)
    S, _ = simular_heston(s0=s0, v0=p["v0"], mu=p["mu"], kappa=p["kappa"],
                          theta=p["theta"], xi=p["xi"], rho=p["rho"], horizonte=H)
    media = S.mean(0); p5 = np.percentile(S, 5, 0); p95 = np.percentile(S, 95, 0)
    fsim = fechas_futuras(corte_ts, H)
    L = min(len(fsim), len(media)); fsim, media, p5, p95 = fsim[:L], media[:L], p5[:L], p95[:L]
    real = dte[col].values; freal = dte.index
    fin = fsim[-1]

    fig, ax = plt.subplots(figsize=(11, 4.7))
    hist = dtr[col].iloc[-ZOOM_BT:]
    ax.plot(hist.index, hist.values, color=C_HIST, lw=1.2, label="Histórico")

    # rectas de tendencia (se prolongan hasta el final de la predicción)
    for dias, c, etq in [(126, C_T6M, "6 meses"), (252, C_T1A, "1 año"),
                         (len(dtr), C_TTOT, "Todo el histórico")]:
        sub = dtr[col].iloc[-dias:]
        a, b, pend = _tendencia(sub)
        nptos = len(sub) + L  # puntos desde el inicio de la ventana hasta el fin de la sim
        ax.plot([sub.index[0], fin], [b, a * (nptos - 1) + b],
                ls="--", lw=1.5, color=c, label=f"Tendencia {etq} ({pend:+.1f}%/año)")

    ax.fill_between(fsim, p5, p95, color=C_BANDA, alpha=0.12, label="IC 90%")
    ax.plot(fsim, media, color=C_PRED, lw=2.3, label="Heston (predicción)")
    ax.plot(freal, real, color=C_REAL, lw=2.4, label="Precio real")
    ax.axvline(corte_ts, color="gray", ls="--", lw=1.3, alpha=0.7)

    vis = np.concatenate([hist.values, real, media, p5, p95])
    lo, hi = float(vis.min()), float(vis.max()); m = (hi - lo) * 0.07
    ax.set_ylim(lo - m, hi + m); ax.set_xlim(hist.index[0], fin)

    idx = min(len(real), len(media)) - 1
    err = abs(media[idx] - real[-1]) / real[-1] * 100
    ax.set_title(f"Backtesting — {nombre.upper()}  ·  corte {CORTE}  ·  "
                 f"S₀={s0:.2f} | Real={real[-1]:.2f} | Sim={media[idx]:.2f} | Err={err:.1f}%")
    ax.set_xlabel("Fecha"); ax.set_ylabel("Precio")
    ax.legend(loc="upper left", ncol=2, framealpha=0.9); ax.grid(alpha=0.3)
    try: _formato_fechas(ax)
    except Exception: pass
    fig.tight_layout(); fig.savefig(os.path.join(CARPETA_FINAL, f"bt_{nombre}.png")); plt.close(fig)
    print(f"  final/bt_{nombre}.png   (err {err:.1f}%)")

# =====================================================================
#  2) PREDICCIÓN A FUTURO — S&P 500, 4 horizontes, vista ampliada (2x2)
# =====================================================================
nombre = "sp500"
df = cargar_datos(nombre); col = _col(df)
p = calibrar_red_neuronal(df)
s0 = float(df[col].iloc[-1]); hoy = df.index[-1]

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
fig.suptitle(f"Predicción a futuro — {nombre.upper()}   (calibración por red neuronal)",
             fontsize=13, fontweight="bold")
for ax, (titulo, H) in zip(axes.ravel(), HORIZONTES_FUTURO.items()):
    S, _ = simular_heston(s0=s0, v0=p["v0"], mu=p["mu"], kappa=p["kappa"],
                          theta=p["theta"], xi=p["xi"], rho=p["rho"], horizonte=H)
    media = S.mean(0); p5 = np.percentile(S, 5, 0); p95 = np.percentile(S, 95, 0)
    fsim = fechas_futuras(hoy, H)
    L = min(len(fsim), len(media)); fsim, media, p5, p95 = fsim[:L], media[:L], p5[:L], p95[:L]

    hist = df[col].iloc[-DIAS_ZOOM:]
    ax.plot(hist.index, hist.values, color=C_HIST, lw=1.4, label="Histórico (6 meses)")
    ax.fill_between(fsim, p5, p95, color=C_BANDA, alpha=0.15, label="IC 90%")
    ax.plot(fsim, media, color=C_PRED, lw=2.3, label="Heston (predicción)")
    ax.axvline(hoy, color="gray", ls="--", lw=1.3, alpha=0.7)

    vis = np.concatenate([hist.values, p5, p95])
    lo, hi = float(vis.min()), float(vis.max()); m = (hi - lo) * 0.06
    ax.set_ylim(lo - m, hi + m)
    ax.set_title(f"{titulo}  ·  media {media[-1]:.0f}  [{p5[-1]:.0f}, {p95[-1]:.0f}]", fontsize=10)
    ax.set_xlabel("Fecha"); ax.set_ylabel("Precio"); ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
    try: _formato_fechas_meses(ax)
    except Exception: pass
fig.tight_layout(); fig.savefig(os.path.join(CARPETA_FINAL, f"futuro_{nombre}.png")); plt.close(fig)
print(f"  final/futuro_{nombre}.png")

# =====================================================================
#  3) COMPARATIVO DE TENDENCIAS — pendientes por activo y ventana (a hoy)
# =====================================================================
VENTANAS = {"6 meses": 126, "1 año": 252, "Total": None}
C_VENT   = {"6 meses": "#3b76b0", "1 año": "#f08c00", "Total": "#2e9e5b"}
datos = {v: [] for v in VENTANAS}
for nombre in ACTIVOS:
    df = cargar_datos(nombre); precios = df[_col(df)].dropna()
    for v, dias in VENTANAS.items():
        sub = precios if dias is None else precios.iloc[-dias:]
        datos[v].append(_tendencia(sub)[2])

x = np.arange(len(ACTIVOS)); w = 0.26
fig, ax = plt.subplots(figsize=(10, 5))
for i, (v, vals) in enumerate(datos.items()):
    ax.bar(x + (i - 1) * w, vals, w, label=v, color=C_VENT[v])
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([a.upper() for a in ACTIVOS])
ax.set_ylabel("Pendiente anualizada (%/año)")
ax.set_title("Comparación de pendientes anualizadas por activo y horizonte")
ax.legend(title="Ventana"); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(CARPETA_FINAL, "tend_comparativo.png")); plt.close(fig)
print("  final/tend_comparativo.png")

print("\nListo. Las 6 figuras del Capítulo 5 están en la carpeta 'final/'.")

In [ ]:
# =====================================================================
#  4) COMPARACIÓN REGRESIÓN vs RED — backtesting de mayo, predicciones superpuestas
#     (demuestra de un vistazo que los dos métodos predicen lo mismo)
# =====================================================================
C_REG = "#2166ac"   # azul    -> predicción regresión lineal
C_RN  = "#e8590c"   # naranja -> predicción red neuronal
for nombre in ACTIVOS:
    df  = cargar_datos(nombre); col = _col(df)
    dtr = df.loc[df.index <  corte_ts]
    dte = df.loc[df.index >= corte_ts]
    s0 = float(dtr[col].iloc[-1]); H = len(dte)
    real = dte[col].values; freal = dte.index
    fsim = fechas_futuras(corte_ts, H)

    sims = {}
    for etq, calib in [("reg", calibrar_regresion_lineal), ("rn", calibrar_red_neuronal)]:
        p = calib(dtr)
        S, _ = simular_heston(s0=s0, v0=p["v0"], mu=p["mu"], kappa=p["kappa"],
                              theta=p["theta"], xi=p["xi"], rho=p["rho"], horizonte=H)
        sims[etq] = (S.mean(0), np.percentile(S, 5, 0), np.percentile(S, 95, 0))

    L = min(len(fsim), len(sims["reg"][0]), len(sims["rn"][0])); fsim = fsim[:L]
    m_reg, lo_reg, hi_reg = (a[:L] for a in sims["reg"])
    m_rn,  lo_rn,  hi_rn  = (a[:L] for a in sims["rn"])

    fig, ax = plt.subplots(figsize=(11, 4.7))
    hist = dtr[col].iloc[-ZOOM_BT:]
    ax.plot(hist.index, hist.values, color=C_HIST, lw=1.2, label="Histórico")
    ax.fill_between(fsim, lo_reg, hi_reg, color=C_REG, alpha=0.08)
    ax.plot(fsim, m_reg, color=C_REG, lw=3.0, label="Regresión lineal")
    ax.plot(fsim, m_rn,  color=C_RN,  lw=1.8, ls="--", label="Red neuronal")
    ax.plot(freal, real, color=C_REAL, lw=2.2, label="Precio real")
    ax.axvline(corte_ts, color="gray", ls="--", lw=1.3, alpha=0.7)

    vis = np.concatenate([hist.values, real, m_reg, m_rn])
    lo, hi = float(vis.min()), float(vis.max()); mar = (hi - lo) * 0.08
    ax.set_ylim(lo - mar, hi + mar); ax.set_xlim(hist.index[0], fsim[-1])

    i = min(len(real), L) - 1
    e_reg = abs(m_reg[i] - real[-1]) / real[-1] * 100
    e_rn  = abs(m_rn[i]  - real[-1]) / real[-1] * 100
    ax.set_title(f"Comparación de métodos — {nombre.upper()}  ·  corte {CORTE}  ·  "
                 f"error regresión {e_reg:.1f}%  vs  red {e_rn:.1f}%")
    ax.set_xlabel("Fecha"); ax.set_ylabel("Precio")
    ax.legend(loc="upper left", framealpha=0.9); ax.grid(alpha=0.3)
    try: _formato_fechas(ax)
    except Exception: pass
    fig.tight_layout(); fig.savefig(os.path.join(CARPETA_FINAL, f"cmp_{nombre}.png")); plt.close(fig)
    print(f"  final/cmp_{nombre}.png   (reg {e_reg:.1f}%  ·  red {e_rn:.1f}%)")

In [ ]:
# =====================================================================
#  2) PREDICCIÓN A FUTURO — los CUATRO activos, rejilla 2x2 por activo
# =====================================================================
for nombre in ACTIVOS:
    df = cargar_datos(nombre); col = _col(df)
    p = calibrar_red_neuronal(df)
    s0 = float(df[col].iloc[-1]); hoy = df.index[-1]

    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
    fig.suptitle(f"Predicción a futuro — {nombre.upper()}   (calibración por red neuronal)",
                 fontsize=13, fontweight="bold")
    for ax, (titulo, H) in zip(axes.ravel(), HORIZONTES_FUTURO.items()):
        S, _ = simular_heston(s0=s0, v0=p["v0"], mu=p["mu"], kappa=p["kappa"],
                              theta=p["theta"], xi=p["xi"], rho=p["rho"], horizonte=H)
        media = S.mean(0); p5 = np.percentile(S, 5, 0); p95 = np.percentile(S, 95, 0)
        fsim = fechas_futuras(hoy, H)
        L = min(len(fsim), len(media)); fsim, media, p5, p95 = fsim[:L], media[:L], p5[:L], p95[:L]

        hist = df[col].iloc[-DIAS_ZOOM:]
        ax.plot(hist.index, hist.values, color=C_HIST, lw=1.4, label="Histórico (6 meses)")
        ax.fill_between(fsim, p5, p95, color=C_BANDA, alpha=0.15, label="IC 90%")
        ax.plot(fsim, media, color=C_PRED, lw=2.3, label="Heston (predicción)")
        ax.axvline(hoy, color="gray", ls="--", lw=1.3, alpha=0.7)

        vis = np.concatenate([hist.values, p5, p95])
        lo, hi = float(vis.min()), float(vis.max()); m = (hi - lo) * 0.06
        ax.set_ylim(lo - m, hi + m)
        ax.set_title(f"{titulo}  ·  media {media[-1]:.0f}  [{p5[-1]:.0f}, {p95[-1]:.0f}]", fontsize=10)
        ax.set_xlabel("Fecha"); ax.set_ylabel("Precio"); ax.grid(alpha=0.3)
        ax.legend(loc="upper left")
        try: _formato_fechas_meses(ax)
        except Exception: pass
    fig.tight_layout(); fig.savefig(os.path.join(CARPETA_FINAL, f"futuro_{nombre}.png")); plt.close(fig)
    print(f"  final/futuro_{nombre}.png")